In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import FOLPSD as FOLPS
import time

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../.."))

# Select the backend before importing folps.py
os.environ["FOLPS_BACKEND"] = "numpy"  #'numpy' or 'jax'

from folps import *
from cosmo_class import *

## FOLPSD (AA)

In [ ]:
#omega_i = Omega_i h² 
omega_b = 0.0224;        #baryons
omega_cdm = 0.12;       #CDM
omega_ncdm = 0.00;    #massive neutrinos 
h = 0.67;                 #h = H0/100
z_pk = 0.01;                 #evaluation redshift
CosmoParams = [z_pk, omega_b, omega_cdm, omega_ncdm, h]

Linear power spectrum at redshift z_pk:

In [ ]:
ps = run_class(h = h, ombh2 = omega_b, omch2 = omega_cdm, omnuh2 = omega_ncdm,
                 As = np.exp(3)/(10**10), ns = 0.9649, z = z_pk)

Omega_mnu = (0.06/(93.14 * h**2))
Omega_m = (omega_cdm + omega_b)/h**2 + Omega_mnu

kwargs = {'z': z_pk, 'h': h, 'Omega_m': Omega_m, #'f0':0.6900011469771554,
              'fnu':Omega_mnu/Omega_m}
    
inputpkT = ps['k'], ps['pk']


Nuisance parameters for the linear power spectrum:

Define a vector with the nuisances parameters, NuisanParams = [b1, b2, bs2, b3nl, alpha0, alpha2, alpha4, ctilde, alphashot0, alphashot2, PshotP]

In [ ]:
#bias parameters
b1 = 1;                 
b2 = 2;                 
bs2 = -4/7*(b1 - 1);        
b3nl = 32/315*(b1 - 1);  

#EFT parameters
alpha0, alpha2, alpha4 = -10,-3,2#-10,-3,2    # PEFT(k) = (alpha0 + alpha2*mu^2+ alpha4*mu^4)  k^2 Plin(k)
ctilde = 0               #NLO counterterm
X_FoG = 1  # uses a Lorentzian Damping 1/(1+x^2), with x = X_FoG f sigma_v mu. 

#Stochatics parameters
# Noise is Pshot = PshotP * ( alphashot0 + alphashot2*(k*mu)**2 )
alphashot0 = 2;          
alphashot2 = 1;            
PshotP = 1/0.002    # =1/barn.  Poissonian shot noise
NuisanParams = [b1, b2, bs2, b3nl, alpha0, alpha2, alpha4, ctilde, alphashot0, alphashot2, PshotP,X_FoG]

#### Computation of $M$ matrices 
**They do not depend on the cosmology**, so they are **computed only one time**. That is, the first time the code is called, it computes the $M$ matrices and stores them for the rest of the runs, which can be of the order of thousands in parameter estimations.

In [ ]:
matrices = FOLPS.Matrices()

In [ ]:
Omfid = 0.31519186799  # for AP set > 0

In [ ]:
# output k_ev
k_ev = np.logspace(np.log10(0.01), np.log10(0.3), num = 100) # array of k_ev in [h/Mpc]

In [ ]:
# Compute 1loop integrals
nonlinear = FOLPS.NonLinear(inputpkT, CosmoParams, EdSkernels=False)

In [ ]:
kh, pkl0, pkl2, pkl4 = FOLPS.RSDmultipoles(k_ev, NuisanParams, Omfid = Omfid, AP=True)

In [ ]:
pkl0_c, pkl2_c = FOLPS.RSDmultipoles_marginalized_const(k_ev, NuisanParams, Omfid = Omfid, AP=True)

In [ ]:
pkl0_i, pkl2_i = FOLPS.RSDmultipoles_marginalized_derivatives(k_ev, NuisanParams, Omfid = Omfid, AP=True)

In [ ]:
pkl0_marg = pkl0_c + (pkl0_i[0] * alpha0 + pkl0_i[1] * alpha2 + pkl0_i[2] * alpha4 + pkl0_i[3] * alphashot0 + pkl0_i[4] * alphashot2)
pkl2_marg = pkl2_c + (pkl2_i[0] * alpha0 + pkl2_i[1] * alpha2 + pkl2_i[2] * alpha4 + pkl2_i[3] * alphashot0 + pkl2_i[4] * alphashot2)

## folps.py (HN)

In [ ]:
kwargs = {'z': z_pk, 
          'h': h, 
          'Omega_m': (omega_b + omega_cdm + omega_ncdm)/h**2, 
          #'f0':0.535274946761771,
          'fnu':omega_ncdm/(omega_b + omega_cdm + omega_ncdm)
         }

In [ ]:
matrix = MatrixCalculator(A_full=False)
mmatrices = matrix.get_mmatrices()

In [ ]:
qpar, qper = qpar_qperp(Omega_fid=Omfid, Omega_m=kwargs['Omega_m'], z_pk=kwargs['z'])

In [ ]:
nonlinear_hn = NonLinearPowerSpectrumCalculator(mmatrices=mmatrices,
                                             kernels='fk',
                                             **kwargs)

In [ ]:
table, table_now = nonlinear_hn.calculate_loop_table(k=inputpkT[0], pklin=inputpkT[1],cosmo=None, **kwargs)

In [ ]:
multipoles = RSDMultipolesPowerSpectrumCalculator(model='FOLPSD') 

In [ ]:
P0, P2, P4  = multipoles.get_rsd_pkell(kobs=k_ev, qpar=qpar, qper=qper, pars=NuisanParams,
                                       table=table, table_now=table_now,
                                       bias_scheme='folps', damping='lor', 
                                       )

In [ ]:
P0_c, P2_c, P4_c  = get_rsd_pkell_marg_const(kobs=k_ev, qpar=qpar, qper=qper, pars=NuisanParams,
                                       table=table, table_now=table_now,
                                       bias_scheme='folps', damping='lor', model = 'FOLPSD'
                                       )

In [ ]:
P0_i, P2_i, P4_i  = get_rsd_pkell_marg_derivatives(kobs=k_ev, qpar=qpar, qper=qper, pars=NuisanParams,
                                       table=table, table_now=table_now,
                                       bias_scheme='folps', damping='lor', model = 'FOLPSD'
                                       )

In [ ]:
P0_marg = P0_c + (P0_i[0] * alpha0 + P0_i[1] * alpha2 + P0_i[2] * alpha4 + P0_i[3] * alphashot0 + P0_i[4] * alphashot2)
P2_marg = P2_c + (P2_i[0] * alpha0 + P2_i[1] * alpha2 + P2_i[2] * alpha4 + P2_i[3] * alphashot0 + P2_i[4] * alphashot2)

In [ ]:
fig, axs = plt.subplots(figsize=(7,5))
axs.set_xlabel(r'$k \, [h\, Mpc^{-1}]$', fontsize =  14)
axs.set_ylabel(r'$\Delta P_{\ell}(k)/P_{\ell}\,\, (\% difference)$', fontsize =  14)

axs.plot(kh,  kh*pkl0, label=r'$\ell = 0$')
axs.plot(kh,  kh*P0, label=r'$folpspy$', ls = '--')

axs.plot(kh,  kh*pkl2, label=r'$\ell = 2$')
axs.plot(kh,  kh*P2, label=r'$folpspy$', ls = '--')

#axs.plot(kh,  kh*pkl4, label=r'$\ell = 0$')
#axs.plot(kh,  kh*P4, label=r'$folpspy$', ls = '--')


axs.plot(kh,  kh*P0_marg, lw=3, ls = ':', color = 'k')
axs.plot(kh,  kh*P2_marg, lw=3, ls = ':', color = 'k')


axs.set_xlim([0, 0.3])
plt.show()

In [ ]:
fig, axs = plt.subplots(figsize=(7,5))
axs.set_xlabel(r'$k \, [h\, Mpc^{-1}]$', fontsize =  14)
axs.set_ylabel(r'$\Delta P_{\ell}(k)/P_{\ell}\,\, (\% difference)$', fontsize =  14)

axs.plot(kh,  kh*pkl0_marg, label=r'$\ell = 0$')
axs.plot(kh,  kh*pkl2_marg, label=r'$\ell = 2$')


#axs.plot(kh, kh * pkl2, label=r'$\ell = 2$')

axs.plot(kh,  kh*P0_marg, lw=3, ls = ':', color = 'k')
axs.plot(kh,  kh*P2_marg, lw=3, ls = ':', color = 'k')


axs.set_xlim([0, 0.3])
plt.show()

In [ ]:
fig, axs = plt.subplots(figsize=(7,5))
axs.set_xlabel(r'$k \, [h\, Mpc^{-1}]$', fontsize =  14)
axs.set_ylabel(r'$\Delta P_{\ell}(k)/P_{\ell}\,\, (\% difference)$', fontsize =  14)

axs.plot(kh,  100*abs(1-pkl0/P0), label=r'$\ell = 0$')
axs.plot(kh,  100*abs(1-pkl2/P2), label=r'$\ell = 2$')
#axs.plot(kh,  100*abs(1-pkl4/P4), label=r'$\ell = 4$')

#axs.plot(kh, kh * pkl2, label=r'$\ell = 2$')



axs.set_xlim([0, 0.3])
plt.show()


In [ ]:
fig, axs = plt.subplots(figsize=(7,5))
axs.set_xlabel(r'$k \, [h\, Mpc^{-1}]$', fontsize =  14)
axs.set_ylabel(r'$\Delta P_{\ell}(k)/P_{\ell}\,\, (\% difference)$', fontsize =  14)

axs.plot(kh,  100*abs(1-pkl0_marg/P0_marg), label=r'$\ell = 0$')
axs.plot(kh,  100*abs(1-pkl2_marg/P2_marg), label=r'$\ell = 2$')
#axs.plot(kh,  100*abs(1-pkl4/P4), label=r'$\ell = 4$')

#axs.plot(kh, kh * pkl2, label=r'$\ell = 2$')



axs.set_xlim([0, 0.3])
plt.show()

# Run Bispectrum

In [ ]:
# niusance parameters for the bispectrum
Pshot = 0; # This would be the same as PshopP*alphashot0 if the bispectrum were computed up to 1loop (I think!)
Bshot = 0;
c1=0
c2=0
X_FoG_bk=0

bisp_nuis_params = [b1, b2, bs2, c1,c2,Pshot,Bshot, X_FoG_bk]
bisp_cosmo_params = [(omega_cdm+omega_b+omega_ncdm)/h**2,h]

In [ ]:
k_ev = np.logspace(np.log10(0.01), np.log10(0.3), num = 30) # array of k_ev in [h/Mpc]
k_ev_bk=np.vstack([k_ev,k_ev]).T   # List of pairs of k. (B=B(k1,k2))

In [ ]:
Omfid = -1 # for AP

In [ ]:
k_pkl_pklnw=np.array([nonlinear[0][0], nonlinear[0][1], nonlinear[1][1]])
B000, B202 = FOLPS.Bisp_Sugiyama(bisp_cosmo_params, bisp_nuis_params,
                                   k_pkl_pklnw=k_pkl_pklnw, z_pk=z_pk, k1k2pairs=k_ev_bk,
                                   Omfid=Omfid,precision=[10,8,8])

In [ ]:
FOLPS.f0

In [ ]:
bispectrum = BispectrumCalculator(basis='sugiyama', model = 'EFT')

b000, b202 = bispectrum.Bisp_Sugiyama(f=FOLPS.f0,  bpars=bisp_nuis_params,
                                      k_pkl_pklnw=k_pkl_pklnw, z_pk=kwargs['z'],
                                      k1k2pairs=k_ev_bk, qpar=1, qperp=1, precision=[10,8,8], damping=None)

In [ ]:
fig, axs = plt.subplots(figsize=(7,5))
axs.set_xlabel(r'$k \, [h\, Mpc^{-1}]$', fontsize =  14)
axs.set_ylabel(r'$k^2 B(k,k) \, [h^{-1} \,  Mpc]^4$', fontsize =  14)

axs.plot(k_ev, k_ev**2 * B000, label=r'B000')
axs.plot(k_ev, k_ev**2 * B202,label=r'B202')

axs.plot(k_ev, k_ev**2 * b000, ls ='--')
axs.plot(k_ev, k_ev**2 * b202, ls ='--')

axs.set_xlim([0, 0.20])
plt.show()

In [ ]:
fig, axs = plt.subplots(figsize=(7,5))
axs.set_xlabel(r'$k \, [h\, Mpc^{-1}]$', fontsize =  14)
axs.set_ylabel(r'$\Delta P_{\ell}(k)/P_{\ell}\,\, (\% difference)$', fontsize =  14)

axs.plot(k_ev,  100*abs(1-b000/B000))
axs.plot(k_ev,  100*abs(1-b202/B202))

#axs.plot(kh, kh * pkl2, label=r'$\ell = 2$')



axs.set_xlim([0, 0.3])
plt.show()